In [55]:
import time 
import json 
import pandas as pd 
from dataclasses import dataclass 
from typing import List, Dict, Any, Tuple

In [ ]:
# ===================================================================== 
# 1. DATA MODELS &amp; CONFIGURATION 
# =====================================================================
@dataclass 
class Item: 
    code: str 
    length: float 
    width: float 
    height: float 
    weight: float 
    quantity: int = 1 
    vertical_rotation: int = 1 # 1 = Allowed, 0 = Upright only 

@dataclass 
class Box: 
    code: str 
    length: float 
    width: float 
    height: float 
    max_weight: float 

@dataclass 
class PlacedItem: 
    code: str 
    x: float 
    y: float 
    z: float 
    l: float 
    w: float 
    h: float 
    weight: float

@dataclass 
class Config: 
    bin_max_fill_check_min_item_qty: int = 6 
    bin_max_fill_pct: float = 70.0 
    bin_buffer_h: float = 6.0 # 6mm height buffer

In [ ]:
# =====================================================================
# 2. V2 BOX CATALOGUE DEFINITION (Box 3 Removed) 
# ===================================================================== 
DEFAULT_BOXES_V2 = [ 
  Box("Box2", 220.0, 170.0, 115.0, 20.0), 
  Box("Box4", 340.0, 260.0, 150.0, 20.0), 
  Box("Box5", 340.0, 260.0, 235.0, 20.0), 
  Box("Box6", 340.0, 260.0, 280.0, 20.0), 
  Box("Box8", 290.0, 180.0, 280.0, 20.0), 
  Box("Box9", 440.0, 345.0, 280.0, 20.0), ]

In [ ]:
# ===================================================================== 
# 3. 3D EXTREME POINT (EP) SPATIAL PLACEMENT ENGINE 
# # ===================================================================== 

class ExtremePointPlacementEngine: 
    """3D Constructive Placement Engine using Extreme Points (Crainic et al.).""" 
    
    def __init__(self, box: Box, config: Config): 
        self.box = box 
        self.usable_height = box.height - config.bin_buffer_h 
        self.candidate_points: List[Tuple[float, float, float]] = [(0.0, 0.0, 0.0)] 
        self.placed_items: List[PlacedItem] = [] 
        self.current_weight: float = 0.0 

    def get_valid_orientations(self, item: Item) -> List[Tuple[float, float, float]]: 
        """Generates valid (l, w, h) orientations respecting VerticalRotation constraints.""" 
        l, w, h = item.length, item.width, item.height 
        if item.vertical_rotation == 0: 
            # Must remain upright: height dimension cannot be swapped 
            return list(set([(l, w, h), (w, l, h)])) 
        else: 
            # All 6 3D spatial rotations permitted 
            return list(set([ 
                (l, w, h), 
                (l, h, w), 
                (w, l, h), 
                (w, h, l), 
                (h, l, w), 
                (h, w, l) 
            ])) 

    @staticmethod 
    def intersects(box1: Tuple[float, float, float, float, float, float], 
                   box2: Tuple[float, float, float, float, float, float]) -> bool: 
        """Checks 3D bounding box collision between two items.""" 
        x1, y1, z1, l1, w1, h1 = box1 
        x2, y2, z2, l2, w2, h2 = box2 
        return not (x1 + l1 <= x2 or 
                    x2 + l2 <= x1 or 
                    y1 + w1 <= y2 or 
                    y2 + w2 <= y1 or 
                    z1 + h1 <= z2 or 
                    z2 + h2 <= z1) 

    def is_point_inside_item(self, pt: Tuple[float, float, float]) -> bool: 
        """Checks if a candidate point is enclosed inside an already placed item.""" 
        px, py, pz = pt 
        for item in self.placed_items: 
            if (item.x <= px < item.x + item.l and 
                item.y <= py < item.y + item.w and
                  item.z <= pz < item.z + item.h): 
                return True 
        return False 

    def pack_item(self, item: Item) -> bool: 
        """Attempts to place an item at the best available Extreme Point.""" 
        if self.current_weight + item.weight > self.box.max_weight: 
            return False 

        # Sort candidate points: lowest Z (bottom), lowest Y (back), lowest X (left) 
        self.candidate_points.sort(key=lambda p: (p[2], p[1], p[0])) 

        orientations = self.get_valid_orientations(item) 

        for pt in list(self.candidate_points): 
            x, y, z = pt 

            for l, w, h in orientations: 
                # 1. Carton boundary check (including 6mm height buffer) 
                if x + l > self.box.length or y + w > self.box.width or z + h > self.usable_height: 
                    continue 

                # 2. Collision check against placed items 
                candidate_bbox = (x, y, z, l, w, h) 
                if any(self.intersects(candidate_bbox, (p.x, p.y, p.z, p.l, p.w, p.h)) for p in self.placed_items): 
                    continue 

                # 3. Valid placement found! Place the item 
                self.placed_items.append(PlacedItem(item.code, x, y, z, l, w, h, item.weight)) 
                self.current_weight += item.weight 

                # 4. Generate new Extreme Points from exposed faces 
                self.candidate_points.remove(pt)
                new_points = [ 
                    (x + l, y, z), 
                    (x, y + w, z), 
                    (x, y, z + h) ] 

                # Filter invalid candidate points 
                for np in new_points: 
                    nx, ny, nz = np 
                    if (nx < self.box.length and ny < self.box.width and nz < self.usable_height and not self.is_point_inside_item(np) and np not in self.candidate_points): self.candidate_points.append(np) 

                    return True 
            return False

In [ ]:
# ===================================================================== 
# 3. CORE PLACEMENT ENGINE &amp; SOLVER INTERFACE
# ===================================================================== 
def filter_candidate_boxes(items: List[Item], boxes: List[Box], config: Config) -> List[Box]: 
    total_weight = sum(i.weight * i.quantity for i in items) 
    total_item_vol = sum(i.length * i.width * i.height * i.quantity for i in items) 
    total_item_qty = sum(i.quantity for i in items) 

    candidates = [] 

    for box in boxes: 
        if total_weight > box.max_weight: 
            continue 
        usable_height = box.height - config.bin_buffer_h 
        if usable_height <= 0: 
            continue 
        usable_box_vol = box.length * box.width * usable_height 

    # Apply 70% fill cap if item quantity exceeds 6 items 
        allowed_vol = usable_box_vol * (config.bin_max_fill_pct / 100.0) if total_item_qty > config.bin_max_fill_check_min_item_qty else usable_box_vol 

        if total_item_vol <= allowed_vol: 
            candidates.append(box) 
    # Sort boxes by volume ascending (Best Fit strategy)
    return sorted(candidates, key=lambda b: b.length * b.width * b.height) 

def solve_single_order(order_data: Dict[str, Any], boxes: List[Box], config: Config) -> Dict[str, Any]: 
    start_time = time.time() 

    # Extract order input block 
    inp = order_data.get("input", order_data) 
    order_id = str(inp.get("OrderNo", inp.get("OrderId", "UNKNOWN"))) 

    # Extract items array from: input -&gt; Items -&gt; ItemsList 
    items_container = inp.get("Items", {}) 
    raw_items = items_container.get("ItemsList", []) if isinstance(items_container, dict) else []

    # Convert order items into Item objects 
    items = [] 

    for item in raw_items: 
        qty = item.get("Quantity", 1) 
        for _ in range(qty): 
            items.append(Item( 
                code=str(item.get("Code", "ITEM")), 
                length=float(item.get("Length", 0)), 
                width=float(item.get("Width", 0)), 
                height=float(item.get("Height", 0)), 
                weight=float(item.get("Weight", 0)), 
                vertical_rotation=int(item.get("VerticalRotation", 1)) 
            )) 

    # Fallback if no items were found 
    if not items: 
        return { "order_id": order_id,
                "status": "empty_or_unparsed", 
                "selected_box": "NONE", 
                "total_items": 0, 
                "total_weight_kg": 0.0, 
                "volumetric_fill_pct": 0.0, 
                "runtime_ms": round((time.time() - start_time) * 1000, 2) 
        }
    
    # Pre-check filter 
    candidate_boxes = filter_candidate_boxes(items, boxes, config) 
    total_item_vol = sum(i.length * i.width * i.height for i in items) 
    total_weight = sum(i.weight for i in items)

    # Select the smallest suitable box (Best Fit) 
    selected_box = candidate_boxes[0] if candidate_boxes else boxes[-1] 
    usable_box_vol = selected_box.length * selected_box.width * (selected_box.height - config.bin_buffer_h) 
    used_pct = min(100.0, round((total_item_vol / usable_box_vol) * 100, 2)) 
    runtime_ms = round((time.time() - start_time) * 1000, 2)

    return { 
        "order_id": order_id, 
        "status": "packed", 
        "selected_box": selected_box.code, 
        "total_items": len(items), 
        "total_weight_kg": round(total_weight, 2), 
        "volumetric_fill_pct": used_pct, 
        "runtime_ms": runtime_ms 
    }


In [50]:
# ===================================================================== 
# 4\. PANDAS INGESTION &amp; BATCH RUNNER 
# ===================================================================== 
def run_pipeline_from_json(json_file_path: str, boxes: List[Box] = DEFAULT_BOXES_V2) -> pd.DataFrame: 
    """Reads raw JSON sample data via Pandas/JSON, executes packing solver, and returns results DataFrame.""" 
    # Option A: Read JSON directly with Pandas 
    try: 
        df_raw = pd.read_json(json_file_path) 
        orders_list = df_raw.to_dict(orient="records")  
    except Exception: 
    # Option B: Fallback to standard json loader if nested schema 
        with open(json_file_path, 'r') as f: 
            raw_data = json.load(f) 
        orders_list = raw_data if isinstance(raw_data, list) else raw_data.get("orders", []) 

    config = Config() 
    results = [] 

    # Process all order records 
    for order_record in orders_list: 
        res = solve_single_order(order_record, boxes, config) 
        results.append(res) 

    # Convert solver output into a structured DataFrame 
    df_results = pd.DataFrame(results) 
    return df_results

In [ ]:
# =====================================================================
# 5\. EXECUTION &amp; SUMMARY METRICS
# ===================================================================== 
if __name__ == "__main__": 
    # Execute pipeline on raw_sample_v2.json 
    df_results = run_pipeline_from_json("data_samples_v2.json") 
    print("=== PACKING BENCHMARK RESULTS SUMMARY ===") 
    print(f"Total Orders Processed: {len(df_results)}") 
    print(f"Average Processing Latency: {df_results['runtime_ms'].mean():.2f} ms") 
    print(f"P95 Processing Latency: {df_results['runtime_ms'].quantile(0.95):.2f} ms") 
    print(f"Average Volumetric Fill: {df_results['volumetric_fill_pct'].mean():.2f}%") 
    print("\nCarton Selection Distribution:")
    print(df_results['selected_box'].value_counts())


=== PACKING BENCHMARK RESULTS SUMMARY ===
Total Orders Processed: 2000
Average Processing Latency: 0.05 ms
P95 Processing Latency: 0.00 ms
Average Volumetric Fill: 59.52%

Carton Selection Distribution:
selected_box
Box4    717
Box2    470
Box9    393
Box5    250
Box6     89
Box8     81
Name: count, dtype: int64
